# Title: 01_feature_extraction
### Goal: Build _data/processed/features.csv_ from ENCODE bigWig trakcs over safe harbor + matched control windows.

In [35]:
from pathlib import Path
from tqdm import tqdm
import random
import numpy as np
import pandas as pd
import requests
import pyranges as pr
import gzip
import pyBigWig

PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT

PosixPath('/home/mazani/projects/safe-harbor-ml')

## Define anchor loci (hg38)

In [36]:
# Known safe harbor anchor regions (hg38)
# These serve as starting points for fixed-size windows

positives = pd.DataFrame(
    [
      # name, chromosome, start end (hg38)
        ("AAVS1_PPP1R12C", "chr19", 55115996, 55115997),
        ("CCR5", "chr3", 46369996, 46371554),
        ("ROSA26_like_THUMPD3_AS1", "chr3", 9349689, 9398625),
    ],
    columns=["locus", "Chromosome", "Start", "End"]
    )

positives

,locus,Chromosome,Start,End
0,AAVS1_PPP1R12C,chr19,55115996,55115997
1,CCR5,chr3,46369996,46371554
2,ROSA26_like_THUMPD3_AS1,chr3,9349689,9398625


## Convert anchor loci into fixed-size genomic windows

In [37]:
WINDOW_BP = 2000 # 2 kb window

def anchors_to_windows(df: pd.DataFrame, window_bp: int) -> pd.DataFrame:
    """
    Convert anchor regions into fixed-size windows centered on the anchor midpoint.
    """

    out = df.copy()

    midpoints = ((out["Start"] + out["End"]) // 2).astype(int)
    half = window_bp // 2

    out["Start"] = (midpoints - half).clip(lower=0)
    out["End"] = midpoints + half

    return out

pos_windows = anchors_to_windows(positives, WINDOW_BP)
pos_windows["label"] = 1
pos_windows["tier"] = "T1"

pos_windows

,locus,Chromosome,Start,End,label,tier
0,AAVS1_PPP1R12C,chr19,55114996,55116996,1,T1
1,CCR5,chr3,46369775,46371775,1,T1
2,ROSA26_like_THUMPD3_AS1,chr3,9373157,9375157,1,T1


## Downlaod/load ENCODE hg38 blacklist

In [38]:
blacklist_path = DATA_RAW / "hg38-blacklist.bed"

if not blacklist_path.exists():
    url = "https://www.encodeproject.org/files/ENCFF356LFX/@@download/ENCFF356LFX.bed.gz"
    gz_path = DATA_RAW / "ENCFF356LFX.bed.gz"

    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    with open(gz_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            if chunk:
                f.write(chunk)

    with gzip.open(gz_path, "rt") as fin, open(blacklist_path, "w") as fout:
        for line in fin:
            fout.write(line)

blacklist = pr.read_bed(str(blacklist_path))
blacklist

,Chromosome,Start,End
0,chr1,628903,635104
1,chr1,5850087,5850571
2,chr1,8909610,8910014
3,chr1,9574580,9574997
4,chr1,32043823,32044203
...,...,...,...
905,chrY,11290797,11334278
906,chrY,11493053,11592850
907,chrY,11671014,11671046
908,chrY,11721528,11749472


## Convert positives to PyRanges

In [6]:
pos_gr = pr.PyRanges(pos_windows[["Chromosome", "Start", "End", "locus", "label", "tier"]])
pos_gr

,Chromosome,Start,End,locus,label,tier
0,chr3,46369775,46371775,CCR5,1,T1
1,chr3,9373157,9375157,ROSA26_like_THUMPD3_AS1,1,T1
2,chr19,55114996,55116996,AAVS1_PPP1R12C,1,T1


## Generate Matched Negatives

In [7]:
NEG_PER_POS_FINAL = 300
FLANK_BP = 5_000_000  # sample within +/- 5 Mb of each positive (keeps chrom context somewhat similar)

def make_negative_pool(pos_df, n_per_pos=1000, flank_bp=5_000_000, window_bp=2000):
    half = window_bp // 2
    rows = []

    for _, row in pos_df.iterrows():
        chrom = row["Chromosome"]
        mid = (row["Start"] + row["End"]) // 2

        for _ in range(n_per_pos):
            new_mid = mid + random.randint(-flank_bp, flank_bp)
            new_mid = max(new_mid, half + 1)
            rows.append((chrom, new_mid - half, new_mid + half))

    neg = pd.DataFrame(rows, columns=["Chromosome", "Start", "End"])
    neg["label"] = 0
    neg["tier"] = "NEG"
    neg["locus"] = "NEG"
    return neg

neg_pool = make_negative_pool(pos_windows, n_per_pos=2000, flank_bp=FLANK_BP, window_bp=WINDOW_BP)
neg_gr = pr.PyRanges(neg_pool)

# Remove overlaps with blacklist and positives
neg_gr = neg_gr.subtract(blacklist).subtract(pos_gr)

# subtract can fragment intervals; keep only intervals that still match the intended window length
neg_df = neg_gr.df
neg_df["len"] = neg_df["End"] - neg_df["Start"]
neg_df = neg_df[(neg_df["len"] >= int(WINDOW_BP * 0.95)) & (neg_df["len"] <= int(WINDOW_BP * 1.05))].drop(columns=["len"])

# Sample final negatives
target_neg = len(pos_windows) * NEG_PER_POS_FINAL
neg_final = neg_df.sample(n=min(target_neg, len(neg_df)), random_state=SEED).reset_index(drop=True)

# Combine
windows = pd.concat(
    [
        pos_windows[["Chromosome", "Start", "End", "locus", "label", "tier"]],
        neg_final[["Chromosome", "Start", "End", "locus", "label", "tier"]],
    ],
    ignore_index=True
)

windows.shape, windows["label"].value_counts()

((903, 6),
 label
 0    900
 1      3
 Name: count, dtype: int64)

## Search ENCODE for bigWIG tracks (Cell Line: K562)

In [15]:
url = "https://www.encodeproject.org/search/?type=File&format=json&limit=1"
r = requests.get(url, headers={"accept": "application/json"}, timeout=30)

print("Status:", r.status_code)
print("Keys:", list(r.json().keys())[:10])

Status: 200
Keys: ['@context', '@graph', '@id', '@type', 'all', 'clear_filters', 'columns', 'facet_groups', 'facets', 'filters']


#### Search Experiments (K562 + target)

In [22]:
HEADERS = {"accept": "application/json"}

def encode_search_experiments(biosample_term: str, assay_title: str, target_label: str | None = None, limit: int = 10):
    params = {
        "type": "Experiment",
        "format": "json",
        "frame": "object",   # richer objects in @graph
        "limit": limit,
        "status": "released",
        "biosample_ontology.term_name": biosample_term,
        "assay_title": assay_title,
    }
    if target_label is not None:
        params["target.label"] = target_label

    r = requests.get("https://www.encodeproject.org/search/", headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def list_experiment_accessions(search_json: dict) -> pd.DataFrame:
    rows = []
    for obj in search_json.get("@graph", []):
        rows.append({
            "accession": obj.get("accession"),
            "assay_title": obj.get("assay_title"),
            "description": obj.get("description"),
        })
    return pd.DataFrame(rows)

# NOTE: Histone marks are typically under "Histone ChIP-seq" in ENCODE searches. :contentReference[oaicite:3]{index=3}

#### Return 1-3 Experimetns per mark

In [23]:
# NOTE: Histone marks are typically under "Histone ChIP-seq" in ENCODE searches. :contentReference[oaicite:3]{index=3}

exp_atac = encode_search_experiments("K562", "ATAC-seq", target_label=None, limit=5)
exp_k27ac = encode_search_experiments("K562", "Histone ChIP-seq", target_label="H3K27ac", limit=5)
exp_k4me3 = encode_search_experiments("K562", "Histone ChIP-seq", target_label="H3K4me3", limit=5)

df_atac = list_experiment_accessions(exp_atac).assign(mark="ATAC")
df_k27 = list_experiment_accessions(exp_k27ac).assign(mark="H3K27ac")
df_k4m = list_experiment_accessions(exp_k4me3).assign(mark="H3K4me3")

exp_df = pd.concat([df_atac, df_k27, df_k4m], ignore_index=True)
exp_df

,accession,assay_title,description,mark
0,ENCSR956DNB,ATAC-seq,ATAC-seq on K562 cells stained for GATA1 and s...,ATAC
1,ENCSR859USB,ATAC-seq,ATAC-seq on K562 cells stained for GATA1 and s...,ATAC
2,ENCSR483RKN,ATAC-seq,None,ATAC
3,ENCSR017LGQ,ATAC-seq,ATAC-seq on human cell line K562,ATAC
4,ENCSR068MIW,ATAC-seq,ATAC-seq on human cell line K562,ATAC
5,ENCSR000AKP,Histone ChIP-seq,H3K27ac ChIP-seq on human K562,H3K27ac
6,ENCSR668LDD,Histone ChIP-seq,,H3K4me3
7,ENCSR000EWA,Histone ChIP-seq,H3K4me3 ChIP-seq on human K562,H3K4me3
8,ENCSR000DWD,Histone ChIP-seq,H3K4me3 ChIP-seq on human K562,H3K4me3
9,ENCSR000AKU,Histone ChIP-seq,H3K4me3 ChIP-seq on human K562,H3K4me3


#### Fecth experiment JSON and extract GRCh38 bigWig files

In [24]:
def get_experiment_json(accession: str) -> dict:
    url = f"https://www.encodeproject.org/experiments/{accession}/"
    params = {"format": "json", "frame": "embedded"}  # embedded makes it easier to see file fields
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def extract_bigwigs_from_experiment(exp_json: dict, assembly: str = "GRCh38") -> pd.DataFrame:
    rows = []
    for f in exp_json.get("files", []):
        if f.get("file_format") != "bigWig":
            continue
        if f.get("assembly") != assembly:
            continue
        href = f.get("href")
        rows.append({
            "file_accession": f.get("accession"),
            "output_type": f.get("output_type"),
            "assembly": f.get("assembly"),
            "href": href,
        })
    return pd.DataFrame(rows)

# Example: pick the first experiment for each mark and extract its bigWigs
picked = (
    exp_df.dropna(subset=["accession"])
          .groupby("mark", as_index=False)
          .head(1)
)

picked

bw_tables = []
for _, row in picked.iterrows():
    acc = row["accession"]
    mark = row["mark"]
    exp_json = get_experiment_json(acc)
    bw = extract_bigwigs_from_experiment(exp_json, assembly="GRCh38")
    bw["mark"] = mark
    bw["experiment"] = acc
    bw_tables.append(bw)

bw_df = pd.concat(bw_tables, ignore_index=True) if bw_tables else pd.DataFrame()
bw_df


,file_accession,output_type,assembly,href,mark,experiment
0,ENCFF252EZV,signal p-value,GRCh38,/files/ENCFF252EZV/@@download/ENCFF252EZV.bigWig,ATAC,ENCSR956DNB
1,ENCFF674WKB,fold change over control,GRCh38,/files/ENCFF674WKB/@@download/ENCFF674WKB.bigWig,ATAC,ENCSR956DNB
2,ENCFF252GZO,signal p-value,GRCh38,/files/ENCFF252GZO/@@download/ENCFF252GZO.bigWig,ATAC,ENCSR956DNB
3,ENCFF555ITG,signal p-value,GRCh38,/files/ENCFF555ITG/@@download/ENCFF555ITG.bigWig,ATAC,ENCSR956DNB
4,ENCFF892FEF,fold change over control,GRCh38,/files/ENCFF892FEF/@@download/ENCFF892FEF.bigWig,ATAC,ENCSR956DNB
5,ENCFF798GCW,fold change over control,GRCh38,/files/ENCFF798GCW/@@download/ENCFF798GCW.bigWig,ATAC,ENCSR956DNB
6,ENCFF502GAA,signal p-value,GRCh38,/files/ENCFF502GAA/@@download/ENCFF502GAA.bigWig,H3K27ac,ENCSR000AKP
7,ENCFF705LDU,signal p-value,GRCh38,/files/ENCFF705LDU/@@download/ENCFF705LDU.bigWig,H3K27ac,ENCSR000AKP
8,ENCFF467OGB,signal p-value,GRCh38,/files/ENCFF467OGB/@@download/ENCFF467OGB.bigWig,H3K27ac,ENCSR000AKP
9,ENCFF094XCU,signal p-value,GRCh38,/files/ENCFF094XCU/@@download/ENCFF094XCU.bigWig,H3K27ac,ENCSR000AKP


#### Pick one experiment per mark

In [25]:
picked = (
    exp_df.dropna(subset=["accession"])
         .groupby("mark", as_index=False)
         .head(1)
)
picked

,accession,assay_title,description,mark
0,ENCSR956DNB,ATAC-seq,ATAC-seq on K562 cells stained for GATA1 and s...,ATAC
5,ENCSR000AKP,Histone ChIP-seq,H3K27ac ChIP-seq on human K562,H3K27ac
6,ENCSR668LDD,Histone ChIP-seq,,H3K4me3


#### Extract GRCh38 bigWigs from those experiments

In [26]:
def get_experiment_json(accession: str) -> dict:
    url = f"https://www.encodeproject.org/experiments/{accession}/"
    params = {"format": "json", "frame": "embedded"}
    r = requests.get(url, headers=HEADERS, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

def extract_bigwigs_from_experiment(exp_json: dict, assembly: str = "GRCh38") -> pd.DataFrame:
    rows = []
    for f in exp_json.get("files", []):
        if f.get("file_format") != "bigWig":
            continue
        if f.get("assembly") != assembly:
            continue
        rows.append({
            "file_accession": f.get("accession"),
            "output_type": f.get("output_type"),
            "assembly": f.get("assembly"),
            "href": f.get("href"),
            "file_type": f.get("file_type"),
        })
    return pd.DataFrame(rows)

bw_tables = []
for _, row in picked.iterrows():
    acc = row["accession"]
    mark = row["mark"]
    exp_json = get_experiment_json(acc)
    bw = extract_bigwigs_from_experiment(exp_json, assembly="GRCh38")
    bw["mark"] = mark
    bw["experiment"] = acc
    bw_tables.append(bw)

bw_df = pd.concat(bw_tables, ignore_index=True) if bw_tables else pd.DataFrame()
bw_df.sort_values(["mark", "output_type"]).head(30)

,file_accession,output_type,assembly,href,file_type,mark,experiment
1,ENCFF674WKB,fold change over control,GRCh38,/files/ENCFF674WKB/@@download/ENCFF674WKB.bigWig,bigWig,ATAC,ENCSR956DNB
4,ENCFF892FEF,fold change over control,GRCh38,/files/ENCFF892FEF/@@download/ENCFF892FEF.bigWig,bigWig,ATAC,ENCSR956DNB
5,ENCFF798GCW,fold change over control,GRCh38,/files/ENCFF798GCW/@@download/ENCFF798GCW.bigWig,bigWig,ATAC,ENCSR956DNB
0,ENCFF252EZV,signal p-value,GRCh38,/files/ENCFF252EZV/@@download/ENCFF252EZV.bigWig,bigWig,ATAC,ENCSR956DNB
2,ENCFF252GZO,signal p-value,GRCh38,/files/ENCFF252GZO/@@download/ENCFF252GZO.bigWig,bigWig,ATAC,ENCSR956DNB
3,ENCFF555ITG,signal p-value,GRCh38,/files/ENCFF555ITG/@@download/ENCFF555ITG.bigWig,bigWig,ATAC,ENCSR956DNB
10,ENCFF779QTH,fold change over control,GRCh38,/files/ENCFF779QTH/@@download/ENCFF779QTH.bigWig,bigWig,H3K27ac,ENCSR000AKP
11,ENCFF814HWX,fold change over control,GRCh38,/files/ENCFF814HWX/@@download/ENCFF814HWX.bigWig,bigWig,H3K27ac,ENCSR000AKP
12,ENCFF945XHA,fold change over control,GRCh38,/files/ENCFF945XHA/@@download/ENCFF945XHA.bigWig,bigWig,H3K27ac,ENCSR000AKP
15,ENCFF977KGH,fold change over control,GRCh38,/files/ENCFF977KGH/@@download/ENCFF977KGH.bigWig,bigWig,H3K27ac,ENCSR000AKP


#### Choose 1 bigWig file, default fold-change

In [27]:
preferred_output_types = [
    "fold change over control",
    "signal p-value",
    "signal",
]

def choose_one_per_mark(bw_df: pd.DataFrame) -> pd.DataFrame:
    sub = bw_df.dropna(subset=["href"]).copy()
    if sub.empty:
        return sub
    sub["rank"] = sub["output_type"].apply(lambda x: preferred_output_types.index(x) if x in preferred_output_types else 999)
    sub = sub.sort_values(["mark", "rank"])
    return sub.groupby("mark", as_index=False).head(1)

chosen_bw = choose_one_per_mark(bw_df)
chosen_bw

,file_accession,output_type,assembly,href,file_type,mark,experiment,rank
1,ENCFF674WKB,fold change over control,GRCh38,/files/ENCFF674WKB/@@download/ENCFF674WKB.bigWig,bigWig,ATAC,ENCSR956DNB,0
10,ENCFF779QTH,fold change over control,GRCh38,/files/ENCFF779QTH/@@download/ENCFF779QTH.bigWig,bigWig,H3K27ac,ENCSR000AKP,0
26,ENCFF804OLI,fold change over control,GRCh38,/files/ENCFF804OLI/@@download/ENCFF804OLI.bigWig,bigWig,H3K4me3,ENCSR668LDD,0


#### Download the chosen bigWigs

In [28]:
from pathlib import Path

TRACK_DIR = DATA_RAW / "tracks"
TRACK_DIR.mkdir(parents=True, exist_ok=True)

def encode_download_url(href: str) -> str:
    return "https://www.encodeproject.org" + href

def download_file(url: str, outpath: Path) -> None:
    if outpath.exists() and outpath.stat().st_size > 0:
        return
    r = requests.get(url, stream=True, timeout=180)
    r.raise_for_status()
    with open(outpath, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

local_tracks = []
for _, row in chosen_bw.iterrows():
    url = encode_download_url(row["href"])
    mark = row["mark"]
    facc = row["file_accession"]
    outpath = TRACK_DIR / f"{mark}__{facc}.bigWig"
    download_file(url, outpath)
    local_tracks.append((mark, facc, outpath))

local_tracks


[('ATAC',
  'ENCFF674WKB',
  PosixPath('/home/mazani/projects/safe-harbor-ml/data/raw/tracks/ATAC__ENCFF674WKB.bigWig')),
 ('H3K27ac',
  'ENCFF779QTH',
  PosixPath('/home/mazani/projects/safe-harbor-ml/data/raw/tracks/H3K27ac__ENCFF779QTH.bigWig')),
 ('H3K4me3',
  'ENCFF804OLI',
  PosixPath('/home/mazani/projects/safe-harbor-ml/data/raw/tracks/H3K4me3__ENCFF804OLI.bigWig'))]

## Define bigWig summary function (mean/max + coverage)

In [39]:
def summarize_bigwig_over_windows(bw_path, windows_df):
    """
    Computer summary stats of a bigWig signal over each window.
    Returns: mean, max, coverage (fraction of finite bases)
    """
    bw = pyBigWig.open(str(bw_path))

    means = []
    maxs = []
    coverages = []

    for chrom, start, end in tqdm(
        windows_df[["Chromosome", "Start", "End"]].itertuples(index=False),
        total=len(windows_df),
        desc=bw_path.name
    ):
        try:
            arr = np.array(bw.values(chrom, int(start), int(end), numpy=True))
            finite = np.isfinite(arr)

            if finite.sum() == 0:
                means.append(np.nan)
                maxs.append(np.nan)
                coverages.append(0.0)
            else:
                vals = arr[finite]
                means.append(float(vals.mean()))
                maxs.append(float(vals.max()))
                coverages.append(float(finite.mean()))
        except RuntimeError:
            # e.g., chromosome absent in file
            means.append(np.nan)
            maxs.append(np.nan)
            coverages.append(0.0)
    bw.close()

    return pd.DataFrame({"mean": means, "max":maxs, "coverage": coverages})

#### Compute features in the windows

In [41]:
features = windows.copy()

for mark, file_acc, path in local_tracks:
    stats = summarize_bigwig_over_windows(path, windows)
    prefix = f"{mark}__{file_acc}"
    prefix = f"{mark}__{file_acc}"
    features[f"{prefix}__mean"] = stats["mean"]
    features[f"{prefix}__max"] = stats["max"]
    features[f"{prefix}__coverage"] = stats["coverage"]

features.head()

H3K4me3__ENCFF804OLI.bigWig: 100%|███████████████████████████████████████████████████████████████████| 903/903 [00:00<00:00, 17035.45it/s]


,Chromosome,Start,End,locus,label,tier,ATAC__ENCFF674WKB__mean,ATAC__ENCFF674WKB__max,ATAC__ENCFF674WKB__coverage,H3K27ac__ENCFF779QTH__mean,H3K27ac__ENCFF779QTH__max,H3K27ac__ENCFF779QTH__coverage,H3K4me3__ENCFF804OLI__mean,H3K4me3__ENCFF804OLI__max,H3K4me3__ENCFF804OLI__coverage
0,chr19,55114996,55116996,AAVS1_PPP1R12C,1,T1,0.626169,3.497290,1.0,8.296354,30.482821,1.0,23.634249,60.093651,1.0
1,chr3,46369775,46371775,CCR5,1,T1,0.942753,4.292660,1.0,0.270727,2.512690,1.0,0.782330,2.149200,1.0
2,chr3,9373157,9375157,ROSA26_like_THUMPD3_AS1,1,T1,0.402415,2.146330,1.0,0.358053,2.212360,1.0,0.638809,2.401730,1.0
3,chr3,12908323,12910323,NEG,0,NEG,1.270906,3.550250,1.0,0.750004,2.713980,1.0,0.575931,1.895970,1.0
4,chr3,50258990,50260990,NEG,0,NEG,3.682713,21.712681,1.0,3.129672,17.367559,1.0,6.657393,28.182739,1.0


#### QC

In [42]:
feat_cols = [c for c in features.columns if "__" in c]
missing = features[feat_cols].isna().mean().sort_values(ascending=False)
missing

ATAC__ENCFF674WKB__mean           0.043189
ATAC__ENCFF674WKB__max            0.043189
H3K27ac__ENCFF779QTH__mean        0.043189
H3K27ac__ENCFF779QTH__max         0.043189
H3K4me3__ENCFF804OLI__mean        0.043189
H3K4me3__ENCFF804OLI__max         0.043189
ATAC__ENCFF674WKB__coverage       0.000000
H3K27ac__ENCFF779QTH__coverage    0.000000
H3K4me3__ENCFF804OLI__coverage    0.000000
dtype: float64

#### Save features.csv and confirm file exists

In [44]:
import os

out_csv = DATA_PROCESSED / "features.csv"
features.to_csv(out_csv, index=False)
out_csv

print("Saved:", out_csv)
print("File size (MB):", round(os.path.getsize(out_csv) / (1024*1024), 3))
print("Shape:", features.shape)
print("Label counts:\n", features["label"].value_counts())

Saved: /home/mazani/projects/safe-harbor-ml/data/processed/features.csv
File size (MB): 0.13
Shape: (903, 15)
Label counts:
 label
0    900
1      3
Name: count, dtype: int64
